# Spacecraft - SAC

**VecNormalize** scales the observations and rewards using running statistics, which helps with your large goal bonus.
It keeps those statistics in the vec env, so they are saved to a `.pkl` file next to the model and must be loaded again for evaluation.


In [1]:
pip install mujoco gymnasium stable-baselines3[extra] sympy psutil

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import gymnasium as gym
import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import Spacecraft_V1

**Register the env**

In [3]:
gym.register(
    id="Spacecraft_V1",
    entry_point="Spacecraft_V1:SpacecraftEnv",
    max_episode_steps=2000,
)

**Declare the vectorized env (with VecNormalize) and the model**

In [4]:
# 4 copies of the env, all running in this process
vec_env = make_vec_env("Spacecraft_V1", n_envs=4, vec_env_cls=DummyVecEnv)

# normalize observations and rewards; clip_obs keeps the normalized observations in a sane range
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# gradient_steps=1  -> 1 update per 4 env steps: much faster, but less learning per step
# gradient_steps=-1 -> 1 update per env step: same learning per step as a single env, only a little faster
model = SAC("MlpPolicy", vec_env, verbose=1, device="cpu", gradient_steps=1)

Using cpu device


**Train**

In [5]:
model.learn(total_timesteps=2000000)

----------------------------------
| rollout/           |           |
|    ep_len_mean     | 288       |
|    ep_rew_mean     | -1.37e+03 |
| time/              |           |
|    episodes        | 4         |
|    fps             | 354       |
|    time_elapsed    | 6         |
|    total_timesteps | 2360      |
| train/             |           |
|    actor_loss      | -20.8     |
|    critic_loss     | 1.33      |
|    ent_coef        | 0.845     |
|    ent_coef_loss   | -2.54     |
|    learning_rate   | 0.0003    |
|    n_updates       | 564       |
----------------------------------
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 391       |
|    ep_rew_mean     | -1.56e+03 |
| time/              |           |
|    episodes        | 8         |
|    fps             | 345       |
|    time_elapsed    | 11        |
|    total_timesteps | 4000      |
| train/             |           |
|    actor_loss      | -27.5     |
|    critic_loss    

KeyboardInterrupt: 

**Save the model and the normalization statistics**

In [ ]:
model.save("sac_norm_spacecraft")
vec_env.save("sac_norm_stats.pkl")   # the normalization statistics, needed to evaluate the model
vec_env.close()

**Evaluate model**

In [ ]:
model = SAC.load("sac_norm_spacecraft")
# the model trained on normalized observations, so load the saved statistics to normalize them the same way
stats = VecNormalize.load("sac_norm_stats.pkl", DummyVecEnv([lambda: gym.make("Spacecraft_V1")]))

eval_env = gym.make("Spacecraft_V1", render_mode="human")
obs, info = eval_env.reset()
for _ in range(2000):
    action, _states = model.predict(stats.normalize_obs(obs), deterministic=True)
    obs, reward, terminated, truncated, info = eval_env.step(action)
    if terminated or truncated:
        obs, info = eval_env.reset()
eval_env.close()